In [ ]:
%load_ext autoreload
%autoreload 2

# RAG agent debug

Scratch notebook for poking at the pipeline one step at a time. Run the setup cell once (it loads spaCy, benepar and the embedding model), then rerun any cell below as often as you like.

Kernel: the `nl2sql` conda env, with `ipykernel` installed.

In [ ]:
## agent/ files import each other directly (from agent_utils import ...), so put agent/ on the path
import sys
sys.path.insert(0, "../agent")

from NLU import NLU
from deliberate import deliberate
from NLG import NLG_clarifying_question, NLG_final_answer
import generator_model
from agent_utils import TOP_K, catalog, col_names, embeddings
from RAG_agent import run, ollama_client, bq_client

## Column catalog (what vector search searches over)

In [ ]:
print(f"{len(col_names)} columns, embeddings {embeddings.shape}")
for name, desc in zip(col_names, catalog["col_descs"]):
    print(f"{name}\t{desc}")

## Question

In [ ]:
question = "Which quarter had the highest sales?"

## Step 2: NLU (question segments -> top-k columns)

In [ ]:
segment_topk_dict = NLU(question, TOP_K)
print(segment_topk_dict)

## Steps 3-4: critic picks, sorted into ambiguous / clear / no match

In [ ]:
ambiguous_segments, clear_matches, no_match_segments = deliberate(question, segment_topk_dict)

## Step 5: clarifying question

In [ ]:
print(NLG_clarifying_question(question, ambiguous_segments))

## SQL generation only (no BigQuery run)

In [ ]:
sql = generator_model.get_response(client=ollama_client, question=question, pairs=clear_matches)
print(sql)

## Run the SQL on BigQuery

In [ ]:
rows = list(bq_client.query(sql).result())
for row in rows:
    print(row)

## Step 7: final answer for the user

In [ ]:
print(NLG_final_answer(question, sql, rows))

## Full pipeline (the same code as `python RAG_agent.py`)

If anything is ambiguous, it asks for your answer in an input box.

In [ ]:
answer = run(question)